In [41]:
import json
from pathlib import Path
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np

def load_experiment_results(base_dir="../src/logs"):
    data_list = []
    base_path = Path(base_dir)
    pattern = "*/*/*/version_0/eval/version_0/test_results.json"

    for json_path in base_path.glob(pattern):
        relative_parts = json_path.relative_to(base_path).parts

        loss_combination = relative_parts[0]
        dataset = relative_parts[1]
        weight_map = relative_parts[2]

        # Read the JSON file
        try:
            with open(json_path, "r") as f:
                metrics = json.load(f)

            row = {
                "loss_combination": loss_combination,
                "dataset": dataset,
                "weight_map": weight_map,
                **metrics,  # Flattens the JSON content into the dict
            }
            data_list.append(row)

        except (json.JSONDecodeError, IOError) as e:
            print(f"Error reading {json_path}: {e}")

    df = pd.DataFrame(data_list)
    return df


df = load_experiment_results()

df.set_index(["loss_combination", "dataset", "weight_map"], inplace=True)

In [42]:
df

test/global/dice  test/global/precision  \
loss_combination dataset  weight_map                                            
220110           sbm_mets none                0.535532               0.737262   
                 wmh_wmh  none                0.483420               0.870476   
110000           sbm_mets none                0.465283               0.709394   
                 wmh_wmh  none                0.451166               0.877412   
011101           sbm_mets none                0.529234               0.641199   
                 wmh_wmh  none                0.433582               0.826734   
000110           sbm_mets none                0.491645               0.546034   
                 wmh_wmh  none                0.407955               0.847192   
110110           sbm_mets none                0.554782               0.625803   
                 wmh_wmh  none                0.438713               0.844890   
110101           sbm_mets none                0.540058               0.608895   
                 wmh_wmh  none                0.438608               0.840951   
110220           sbm_mets none                0.508909               0.678924   
                 wmh_wmh  none                0.433181               0.854016   

                                      test/global/recall  test/global/F2  \
loss_combination dataset  weight_map                                       
220110           sbm_mets none                  0.447818        0.485976   
                 wmh_wmh  none                  0.320898        0.367273   
110000           sbm_mets none                  0.443238        0.479195   
                 wmh_wmh  none                  0.318187        0.364672   
011101           sbm_mets none                  0.520578        0.540930   
                 wmh_wmh  none                  0.298701        0.342444   
000110           sbm_mets none                  0.589086        0.579941   
                 wmh_wmh  none                  0.240269        0.280451   
110110           sbm_mets none                  0.605248        0.609250   
                 wmh_wmh  none                  0.292063        0.336038   
110101           sbm_mets none                  0.583197        0.588162   
                 wmh_wmh  none                  0.302468        0.346893   
110220           sbm_mets none                  0.500164        0.527966   
                 wmh_wmh  none                  0.294016        0.338395   

                                      test/cc/dice  test/instance/dice  \
loss_combination dataset  weight_map                                     
220110           sbm_mets none            0.428713            0.549041   
                 wmh_wmh  none            0.250119            0.495494   
110000           sbm_mets none            0.357655            0.458087   
                 wmh_wmh  none            0.180334            0.428539   
011101           sbm_mets none            0.466858            0.594829   
                 wmh_wmh  none            0.260078            0.460552   
000110           sbm_mets none            0.453922            0.572014   
                 wmh_wmh  none            0.245222            0.505951   
110110           sbm_mets none            0.458696            0.542552   
                 wmh_wmh  none            0.217800            0.446121   
110101           sbm_mets none            0.453786            0.568675   
                 wmh_wmh  none            0.251691            0.482577   
110220           sbm_mets none            0.449812            0.557403   
                 wmh_wmh  none            0.225370            0.445539   

                                      test/instance/tp  test/instance/fp  \
loss_combination dataset  weight_map                                       
220110           sbm_mets none                5.071429          1.142857   
                 wmh_wmh  none               23.370370          5.814815   
110000           sbm_mets none     

In [43]:
COLORS = {
    'good': "#6aa84f",
    'bad': "#cc4125",
    'neutral': '#616161',
}

def readable_losses(number_string):
    glob = number_string[:3]
    local = number_string[-3:]
    readablegl = ""
    readablelo = ""
    if glob == '000':
        readablegl += 'none'
    if glob == '220':
        readablegl += '2*DiceCE'
    if glob[0] == '1':
        readablegl += 'Dice'
    if glob[1] == '1':
        readablegl += 'CE'
    if glob[2] == '1':
        readablegl += 'Tversky'
    if local == '000':
        readablelo += 'none'
    if local == '220':
        readablelo += '2*DiceCE'
    if local[0] == '1':
        readablelo += 'Dice'
    if local[1] == '1':
        readablelo += 'CE'
    if local[2] == '1':
        readablelo += 'Tversky'
    return (readablegl, readablelo)

metricreadable = {
  "test/global/dice": "DSC",
  "test/global/F2": "F2",
  "test/instance/f1": "RQ",
  "test/instance/dice": "SQDSC",
  "test/instance/assd": "SQASSD",
  "test/instance/recall":  "recall_inst",
  "test/instance/recall_q0": "recall_inst_Q1",
  "test/instance/recall_q1": "recall_inst_Q2",
  "test/instance/recall_q2": "recall_inst_Q3",
  "test/instance/recall_q3": "recall_inst_Q4",
  "test/cc/dice": "CCDice",
  "test/global/precision": "precision",
  "test/global/recall": "recall",
  "test/instance/precision": "precision_inst",
}

# Loss Baseline Comparisons Lollipop Chart

In [ ]:
def plot_baseline_comparison(
    df, metric, baseline_tuple, comparison_tuples, experiment, save_path
):
    assert experiment in ['loss_combos', 'weight_maps']
    if baseline_tuple not in df.index:
        raise KeyError(
            f"Baseline tuple {baseline_tuple} was not found in the DataFrame index."
        )
    baseline_val = df.loc[baseline_tuple, metric]

    comp_vals = []
    labels = []

    for key in comparison_tuples:
        if key in df.index:
            comp_vals.append(df.loc[key, metric])
            if experiment == 'loss_combos':
                labels.append(f"{readable_losses(key[0])[0]}\n{readable_losses(key[0])[1]}")
            else: 
                labels.append(f"{key[2]}")

    comp_vals = np.array(comp_vals)
    x_positions = np.arange(len(labels))

    fig, ax = plt.subplots(figsize=(7, 6.5))

    ax.axhline(
        y=baseline_val,
        color="#7f8c8d",
        linestyle="--",
        linewidth=1.8,
        label=f"Baseline (DiceCE, none): {baseline_val:.3f}" if experiment == 'loss_combos' else "Baseline ($W_{none}$):" + f"{baseline_val:.3f}",
        zorder=1,
    )

    # 4. Generate vertical lines and markers dynamically
    for x, val, label in zip(x_positions, comp_vals, labels):
        if np.isnan(val):
            continue

        diff = val - baseline_val
        # Soft, clean hex colors for presentation: Green for upgrade, Red for downgrade
        if metricreadable[metric] in ['SQASSD', 'CEDI']:
            color = COLORS['neutral'] if abs(diff) < 0.001 else (COLORS['good'] if diff <= 0 else COLORS['bad'])
        else:
            color = COLORS['neutral'] if abs(diff) < 0.001 else (COLORS['good'] if diff >= 0 else COLORS['bad'])

        # Vertical line indicating distance from baseline
        ax.vlines(
            x,
            ymin=baseline_val,
            ymax=val,
            colors=color,
            linewidth=2.5,
            alpha=0.8,
            zorder=2,
        )

        # Dot marker at the performance value
        ax.scatter(x, val, color=color, s=140, zorder=3)

        # Annotate absolute value and relative change
        va = "bottom" if diff >= 0 else "top"
        offset = 0.005 if diff >= 0 else -0.005
        sign = "+" if diff >= 0 else ""

        ax.text(
            x,
            val + offset,
            f"{val:.3f}\n({sign}{diff:.3f})",
            ha="center",
            va=va,
            fontsize=9.5,
            fontweight="bold",
            color=color,
        )

    # 5. Presentation adjustments & Styling
    ax.set_xticks(x_positions)
    ax.set_xmargin(0.1)
    ax.set_xticklabels(labels, rotation=0, fontsize=10, fontweight="medium")
    ax.set_ylabel(metricreadable[metric], fontsize=12, fontweight="bold", labelpad=10)

    # Add dynamic padding to y-axis limits to prevent text labels from clipping
    valid_vals = [v for v in comp_vals if not np.isnan(v)] + [baseline_val]
    ymin, ymax = min(valid_vals), max(valid_vals)
    ax.set_ylim(ymin - 0.04, ymax + 0.04)

    # Soft background grid lines
    ax.grid(axis="y", linestyle=":", alpha=0.5)
    ax.legend(loc="upper left", frameon=True, facecolor="#f8f9fa", edgecolor="#e2e8f0")

    # Save cleanly with tight formatting layout
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    
    
SUPERSET = 'wmh'
TASK = "wmh"
EXPERIMENT = 'weight_maps'
EXPERIMENT = 'loss_combos'

os.makedirs(f'../docs/thesis/figures/results/{TASK}/lollipop/{EXPERIMENT}/', exist_ok=True)

baseline_config = ("110000", f"{SUPERSET}_{TASK}", "none")
comparison_configs = [
    ("000110", f"{SUPERSET}_{TASK}", "none"),
    ("110110", f"{SUPERSET}_{TASK}", "none"),
    ("220110", f"{SUPERSET}_{TASK}", "none"),
    ("110220", f"{SUPERSET}_{TASK}", "none"),
    ("110101", f"{SUPERSET}_{TASK}", "none"),
    ("011101", f"{SUPERSET}_{TASK}", "none"),
] if EXPERIMENT == 'loss_combos' else [
    ("110000", f"{SUPERSET}_{TASK}", "v_iw"),
    ("110000", f"{SUPERSET}_{TASK}", "v_region"),
    ("110000", f"{SUPERSET}_{TASK}", "v_adaptive"),
    ("110000", f"{SUPERSET}_{TASK}", "v_mountains"),
    ("110000", f"{SUPERSET}_{TASK}", "v_islands"),
]

for metric in metricreadable.keys():
    plot_baseline_comparison(
        df=df,
        metric=metric,
        baseline_tuple=baseline_config,
        comparison_tuples=comparison_configs,
        experiment= EXPERIMENT,
        save_path=f'../docs/thesis/figures/results/{TASK}/lollipop/{EXPERIMENT}/{metricreadable[metric]}',
    )

# Instance Recall by Quartile - Loss Combos

In [47]:
def plot_quartile_baseline_comparison(
    df, baseline_tuple, comparison_tuples, experiment, save_path
):
    assert experiment in ['loss_combos', 'weight_maps']
    quartiles = ["test/instance/recall_q0", "test/instance/recall_q1", "test/instance/recall_q2", "test/instance/recall_q3"]
    quartile_labels = ["Q1", "Q2", "Q3", "Q4"]
    
    # Palette for the four quartiles
    colors = ["#1abc9c", "#3498db", "#9b59b6", "#e67e22"]
    
    if baseline_tuple not in df.index:
        raise KeyError(f"Baseline tuple {baseline_tuple} was not found in the DataFrame index.")
    
    # 1. Extract baselines for all 4 quartiles
    baseline_vals = {q: df.loc[baseline_tuple, q] for q in quartiles}

    # 2. Extract data & build standard labels
    labels = []
    comp_data = {q: [] for q in quartiles}
    
    for key in comparison_tuples:
        if key in df.index:
            if experiment == 'loss_combos':
                labels.append(f"{readable_losses(key[0])[0]}\n{readable_losses(key[0])[1]}")
            else: 
                labels.append(f"{key[2]}")
            for q in quartiles:
                comp_data[q].append(df.loc[key, q])
                
    num_groups = len(labels)
    x_base = np.arange(num_groups)
    
    # 3. Handle Grouping Dimensions
    # total width allowed for all 4 lollipops together per x-tick
    total_group_width = 0.6  
    step = total_group_width / len(quartiles)
    # This centers the group perfectly over the integer x-tick position
    offsets = np.linspace(-total_group_width/2 + step/2, total_group_width/2 - step/2, len(quartiles))

    # 4. Initialize layout (Widened to give 24 lollipops breathing room)
    fig, ax = plt.subplots(figsize=(16, 6.5))

    # Plot baseline dashed lines (shorter, subtle legend tags)
    for idx, q in enumerate(quartiles):
        ax.axhline(
            y=baseline_vals[q],
            color=colors[idx],
            linestyle="--",
            linewidth=1.2,
            alpha=0.6,
            label=f"{quartile_labels[idx]} Base: {baseline_vals[q]:.2f}",
        )

    # 5. Iteratively plot grouped lollipops
    for idx, q in enumerate(quartiles):
        b_val = baseline_vals[q]
        
        for g_idx in range(num_groups):
            val = comp_data[q][g_idx]
            if np.isnan(val):
                continue
                
            # Shift the x position left or right based on which quartile it belongs to
            x_pos = x_base[g_idx] + offsets[idx]
            val = round(val, 3)
            b_val = round(b_val, 3)
            diff = val - b_val
            q_color = COLORS['neutral'] if abs(diff) < 0.005 else (COLORS['good'] if diff >= 0 else COLORS['bad'])
            
            # Draw stem
            ax.vlines(
                x_pos,
                ymin=b_val,
                ymax=val,
                colors=q_color,
                linewidth=2.0,
                alpha=0.8,
                zorder=2,
            )
            
            # Draw marker head
            ax.scatter(x_pos, val, color=q_color, s=70, zorder=3, edgecolors='none')
            
            # Annotation text (smaller size to avoid overcrowding)
            va = "bottom" if diff >= 0 else "top"
            offset = 0.006 if diff >= 0 else -0.006
            sign = "+" if diff >= 0 else ""
            
            ax.text(
                x_pos,
                val + offset,
                f"{sign}{diff:.2f}",
                ha="center",
                va=va,
                fontsize=7.5,
                fontweight="semibold",
                color=q_color,
            )

    # 6. Formatting & Styling
    ax.set_xticks(x_base)
    ax.set_xticklabels(labels, rotation=0, fontsize=10, fontweight="medium")
    ax.set_ylabel("Instance Recall", fontsize=12, fontweight="bold", labelpad=10)
    
    # Push margins slightly so outer lollipops aren't cut off
    ax.set_xmargin(0.08)
    
    # Dynamically find min/max values to safely scale y limits
    all_vals = [v for q in quartiles for v in comp_data[q] if not np.isnan(v)] + list(baseline_vals.values())
    ymin, ymax = min(all_vals), max(all_vals)
    ax.set_ylim(max(0, ymin - 0.06), 1.04)

    ax.grid(axis="y", linestyle=":", alpha=0.5)
    
    # Place Legend cleanly outside or top-left
    ax.legend(loc="upper left", frameon=True, facecolor="#f8f9fa", edgecolor="#e2e8f0", fontsize=9)
    plt.tight_layout()
    #plt.show()
    plt.savefig(save_path, dpi=300)
    plt.close()

SUPERSET = 'wmh'
TASK = "wmh"
EXPERIMENT = 'weight_maps'
EXPERIMENT = 'loss_combos'

baseline_config = ("110000", f"{SUPERSET}_{TASK}", "none")
comparison_configs = [
    ("000110", f"{SUPERSET}_{TASK}", "none"),
    ("110110", f"{SUPERSET}_{TASK}", "none"),
    ("220110", f"{SUPERSET}_{TASK}", "none"),
    ("110220", f"{SUPERSET}_{TASK}", "none"),
    ("110101", f"{SUPERSET}_{TASK}", "none"),
    ("011101", f"{SUPERSET}_{TASK}", "none"),
] if EXPERIMENT == 'loss_combos' else [
    ("110000", f"{SUPERSET}_{TASK}", "v_iw"),
    ("110000", f"{SUPERSET}_{TASK}", "v_region"),
    ("110000", f"{SUPERSET}_{TASK}", "v_adaptive"),
    ("110000", f"{SUPERSET}_{TASK}", "v_mountains"),
    ("110000", f"{SUPERSET}_{TASK}", "v_islands"),
]

# 3. Generate and save the visualization
plot_quartile_baseline_comparison(
    df=df,
    baseline_tuple=baseline_config,
    comparison_tuples=comparison_configs,
    experiment=EXPERIMENT,
    save_path=f'../docs/thesis/figures/results/{TASK}/lollipop/loss_combos/quartile_recall_comparison.png',
)

# Metric Comparison Table - Losses

In [50]:
def generate_typst_comparison_table(
    df, metrics_list, metric_readable, baseline_tuple, comparison_tuples, experiment
):
    assert experiment in ['loss_combos', 'weight_maps']
    headers = [f"[{metric_readable[m]}]" for m in metrics_list]  # Fallback row titles

    num_cols = 1 + 1 + len(comparison_tuples)  # Metric Name + Baseline + Comparisons
    align_str = f"(left, {'center, ' * (num_cols - 1)})".strip(", ")

    typst_lines = [
        '#import "../../../utils.typ": *',
        f'#let {TASK}results-table_{experiment}() = table(',
        f"  columns: (auto, {'1fr, ' * (num_cols - 1)}).slice(0, {num_cols}),",
        f"  align: {align_str},",
        "  stroke: (",
        "    x: none,",
        "    y: 0.4pt + luma(220),",
        "  ),",
        "  inset: 8pt,",
        "",
        "  table.header(",
        "    [],",
    ]

    # Add baseline header (extracting the weight map descriptor name)
    typst_lines.append(f"    [{readable_losses(baseline_tuple[0])}],".replace(')', '').replace('(', '').replace('\'', '') if experiment=='loss_combos' else baseline_tuple[2])

    # Add comparison headers
    for key in comparison_tuples:
        typst_lines.append(f"    [{readable_losses(key[0])}],".replace(')', '').replace('(', '').replace('\'', '') if experiment=='loss_combos' else key[2])
    typst_lines.append("  ),")

    # 2. Process each metric row
    for metric in metrics_list:
        readable_name = metric_readable.get(metric, metric)

        # Handle math formatting variations for specific metrics if needed
        if metric == "F2":
            readable_name = "$F_2$"
        elif metric == "inst_recall":
            readable_name = '$"recall"_"inst"$'

        typst_lines.append(f"\n  // {metric}")
        typst_lines.append(f"  [{readable_name}],")

        # Safely get baseline
        if baseline_tuple not in df.index:
            raise KeyError(f"Baseline {baseline_tuple} not found in DataFrame.")
        baseline_val = df.loc[baseline_tuple, metric]

        # Gather comparison values
        comp_vals = []
        for key in comparison_tuples:
            if key in df.index:
                comp_vals.append(df.loc[key, metric])
            else:
                comp_vals.append(np.nan)

        all_vals = [baseline_val] + comp_vals

        # Determine the "best" score index for bolding
        # Lower is better for SQASSD and CEDI, higher is better for others
        valid_vals = [v for v in all_vals if not np.isnan(v)]
        if not valid_vals:
            continue

        if metric_readable.get(metric, metric) in ["SQASSD", "CEDI"]:
            best_val = min(valid_vals)
        else:
            best_val = max(valid_vals)

        # Render baseline cell
        is_baseline_best = np.isclose(baseline_val, best_val, atol=1e-5)
        b_cell = f"{baseline_val:.3f}"
        if is_baseline_best:
            b_cell = f"#text(size: 10.5pt)[*{b_cell}*]"
        typst_lines.append(f"  [{b_cell}],")

        # Render comparison cells
        for val in comp_vals:
            if np.isnan(val):
                typst_lines.append("  [-],")
                continue

            diff = val - baseline_val
            is_best = np.isclose(val, best_val, atol=1e-5)

            # Format the delta string
            sign = "+" if diff >= 0 else ""
            delta_cell = f"#deltainv({sign}{diff:.3f})" if metric_readable.get(metric, metric) in ["SQASSD", "CEDI"] else f"#delta({sign}{diff:.3f})" 

            # Apply bolding wrapper if this cell is the winner
            if is_best:
                delta_cell = f"#text(size: 10.5pt)[*{delta_cell}*]"

            typst_lines.append(f"  [{delta_cell}],")

    typst_lines.append(")")
    with open(f'../docs/thesis/figures/results/{TASK}/table_{experiment}.typ', 'w') as f:
        f.write("\n".join(typst_lines))
    return "\n".join(typst_lines)


# --- Execution Pipeline ---
SUPERSET = "sbm"
TASK = "mets"
baseline_config = ("110000", f"{SUPERSET}_{TASK}", "none")
comparison_configs = [
    ("000110", f"{SUPERSET}_{TASK}", "none"),
    ("110110", f"{SUPERSET}_{TASK}", "none"),
    ("220110", f"{SUPERSET}_{TASK}", "none"),
    ("110220", f"{SUPERSET}_{TASK}", "none"),
    ("110101", f"{SUPERSET}_{TASK}", "none"),
    ("011101", f"{SUPERSET}_{TASK}", "none"),
]

# Run the function
typst_table_code = generate_typst_comparison_table(
    df=df,
    metrics_list=metricreadable.keys(),
    metric_readable=metricreadable,
    baseline_tuple=baseline_config,
    comparison_tuples=comparison_configs,
    experiment='loss_combos'
)